# Урок 13 - Память агента


## Настройка

В этой тетрадке показано, как создать агента по бронированию путешествий с **постоянной памятью** с использованием **Microsoft Agent Framework** (MAF).

Вы узнаете, как различные типы памяти агента — рабочая, кратковременная и долговременная — влияют на то, как агент сохраняет и использует информацию в ходе разговоров.

**Требования:**
- Проект Microsoft Foundry с развернутой моделью чата (например, `gpt-5-mini`).
- Вход в систему с помощью Azure CLI — выполните `az login` в вашем терминале.
- `AZURE_AI_PROJECT_ENDPOINT` — конечная точка вашего проекта Microsoft Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — имя вашей развернутой модели.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Типы памяти агента

Искусственные агенты могут использовать различные виды памяти, каждая из которых служит своей цели:

### Оперативная память
Сам ход разговора — сообщения, обменянные в рамках одной сессии. Агент может ссылаться на ранее отправленные сообщения в той же цепочке, чтобы сохранять связность. В MAF это создаётся с помощью **`agent.create_session()`**, который возвращает `AgentSession`.

### Кратковременная память
Информация, которая сохраняется в течение выполнения задачи или сессии, но не хранится постоянно. Например, агент может накапливать факты во время многоступенчатого планирования и использовать их для создания итогового маршрута.

### Долговременная память
Предпочтения и факты, которые сохраняются **между сессиями**. Возвращающемуся пользователю не нужно повторять свои диетические ограничения или стиль путешествий. Долговременная память обычно поддерживается внешним хранилищем — базой данных, файлом или векторным индексом — и предоставляется агенту через инструменты.


## Рабочая память с сессиями

Самая простая форма памяти — это сессия разговора. Когда вы передаете одну и ту же сессию (созданную с помощью `agent.create_session()`) в последовательные вызовы `agent.run()`, агент видит всю историю этого разговора и может вспомнить ранее упомянутые детали.

Давайте создадим туристического агента и продемонстрируем рабочую память.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агент правильно вспомнил бюджет, потому что оба сообщения принадлежат одной сессии. Это **рабочая память** — она существует только в течение времени сессии.

### Что происходит с новой нитью?

Если мы создаём **новую** сессию, агент не помнит предыдущий разговор:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Шаблон долговременной памяти

Чтобы запоминать предпочтения пользователя **между сессиями**, нам нужно постоянное хранилище, которое находится вне потока разговора. Агент получает доступ к этому хранилищу через **инструменты** — функции, которые он может вызывать для сохранения и получения информации.

Ниже мы реализуем простое хранилище предпочтений в памяти (в продуктиве вы бы использовали базу данных или векторный индекс) и предоставим его как инструменты, которыми агент может пользоваться.

### Архитектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарий 1 — Пользователь, бронирующий поездку на годовщину в первый раз

Сара посещает сайт впервые. Агент должен сохранить её предпочтения с помощью инструментов и использовать их для рекомендации отелей.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарий 2 — Сара возвращается спустя недели

Сара начинает **совершенно новую сессию** (симуляция нового сеанса). Рабочая память пуста, но в хранилище долгосрочных предпочтений всё ещё есть её информация. Агент должен извлечь её и использовать для персонализации рекомендаций.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

В этом уроке вы узнали о трех типах памяти агента и о том, как их реализовать с помощью Microsoft Agent Framework:

| Тип памяти | Механизм MAF | Время жизни |
|---|---|---|
| **Рабочая** | `agent.create_session()` | Одна беседа |
| **Краткосрочная** | Накопленный контекст внутри потока | Одна задача / сессия |
| **Долгосрочная** | Внешнее хранилище, доступное через функции `@tool` | Между сессиями |

### Основные выводы
1. **`agent.create_session()`** предоставляет рабочую память — агент видит всю историю разговоров в рамках сессии.
2. **Новые сессии теряют контекст** — без долгосрочной памяти агент не может вспомнить прошлые разговоры.
3. **Функции `@tool`** служат связующим звеном — они позволяют агенту сохранять и извлекать информацию из постоянного хранилища.
4. **Персонализация улучшается со временем** — чем больше хранится предпочтений, тем лучше рекомендации агента.

### Применение в реальном мире
- **Обслуживание клиентов**: запомнить историю и предпочтения клиента
- **Персональные помощники**: поддерживать контекст на протяжении дней или недель
- **Здравоохранение**: отслеживать информацию и предпочтения пациентов
- **Электронная коммерция**: персонализированный шопинг на основе истории

### Следующие шаги
- Заменить словарь в памяти на базу данных или векторное хранилище (например, Azure AI Search)
- Добавить срок действия памяти для информации с ограниченным сроком актуальности
- Создавать мультиагентские системы с общей памятью
- Исследовать блокнот Cognee с поддержкой памяти на базе графа знаний


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
